# 09 — 2018 target-pattern diagnostic (Step 5)

This **read-only validation** investigates the large difference between 2018-Q1 and 2018-Q4 24-month composite event rates reported in notebook 08. It uses the existing validated modeling dataset and the existing cleaned 2018 origination cohort. It does not change the target, create drift metrics, train models, or infer a cause from counts alone. Run from the repository root or `notebooks/`.

## 1. Read the existing artifacts

The modeling artifact contains one eligible loan per row with the validated 24-month labels and audit fields. The 2018 cleaned origination artifact supplies the denominator before target eligibility filtering. Stop if either file or an audit field is missing.

In [3]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed" / "Freddie_Mac"
MODEL_PATH = DATA / "freddie_modeling_dataset_2015_2023_24m.parquet"
ORIG_PATH = DATA / "origination_2018_clean.parquet"

required_model = ["SOURCE YEAR", "LOAN IDENTIFIER", "ORIGINATION YEAR",
                  "ORIGINATION QUARTER", "ORIGINATION COHORT", "HORIZON_MONTHS",
                  "TARGET_90_PLUS", "TARGET_COMPOSITE_CREDIT_EVENT",
                  "FIRST_SERIOUS_DELINQUENCY_PERIOD", "FIRST_TERMINAL_CREDIT_EVENT_PERIOD",
                  "FIRST_OBSERVED_MONTH", "LAST_OBSERVED_MONTH", "OBSERVED_MONTHS"]
required_orig = ["LOAN IDENTIFIER", "ORIGINATION YEAR", "ORIGINATION QUARTER",
                 "ORIGINATION COHORT"]
for path in [MODEL_PATH, ORIG_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)
model = pd.read_parquet(MODEL_PATH, columns=required_model)
orig = pd.read_parquet(ORIG_PATH, columns=required_orig)
oot = model[model["SOURCE YEAR"].between(2018, 2023)].copy()
q18 = oot[oot["SOURCE YEAR"].eq(2018)].copy()
print("Model artifact:", MODEL_PATH, "| 2018 eligible rows:", len(q18))
print("2018 cleaned origination:", ORIG_PATH, "| rows:", len(orig))

Model artifact: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\freddie_modeling_dataset_2015_2023_24m.parquet | 2018 eligible rows: 47417
2018 cleaned origination: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\origination_2018_clean.parquet | rows: 50000


## 2. Verify keys, cohorts, labels, and expected 2018 counts

The 2018 quarter totals must match notebook 08 (loans: 11,884 / 11,815 / 11,771 / 11,947; positive events: 63 / 264 / 406 / 489). Report discrepancies and stop before interpreting later tables. The origination file must have one row per identifier and no invalid quarter.

In [4]:
assert len(q18) == 47417, f"Unexpected 2018 modeled count: {len(q18)}"
assert len(orig) == 50000, f"Unexpected 2018 origination count: {len(orig)}"
assert not q18.duplicated(["SOURCE YEAR", "LOAN IDENTIFIER"]).any()
assert not orig["LOAN IDENTIFIER"].duplicated().any()
assert orig["ORIGINATION YEAR"].eq(2018).all()
assert q18["ORIGINATION YEAR"].eq(2018).all()
assert orig["ORIGINATION QUARTER"].isin(["Q1","Q2","Q3","Q4"]).all()
assert q18["ORIGINATION QUARTER"].isin(["Q1","Q2","Q3","Q4"]).all()
assert q18["HORIZON_MONTHS"].eq(24).all()
assert q18[["TARGET_90_PLUS", "TARGET_COMPOSITE_CREDIT_EVENT"]].notna().all().all()
assert q18["TARGET_COMPOSITE_CREDIT_EVENT"].isin([0,1]).all()
assert q18["TARGET_90_PLUS"].isin([0,1]).all()
assert q18["TARGET_90_PLUS"].le(q18["TARGET_COMPOSITE_CREDIT_EVENT"]).all()
cohort_match = q18[["LOAN IDENTIFIER", "ORIGINATION COHORT"]].merge(
    orig[["LOAN IDENTIFIER", "ORIGINATION COHORT"]], on="LOAN IDENTIFIER",
    how="left", validate="one_to_one", indicator=True,
    suffixes=("_MODEL", "_ORIG"))
assert cohort_match["_merge"].eq("both").all()
assert cohort_match["ORIGINATION COHORT_MODEL"].eq(cohort_match["ORIGINATION COHORT_ORIG"]).all()

got = q18.groupby("ORIGINATION QUARTER").agg(
    loans=("LOAN IDENTIFIER", "size"),
    positives=("TARGET_COMPOSITE_CREDIT_EVENT", "sum"))
expected = pd.DataFrame({"loans": [11884,11815,11771,11947],
                         "positives": [63,264,406,489]}, index=pd.Index(["Q1","Q2","Q3","Q4"],name="ORIGINATION QUARTER"))
display(got)
pd.testing.assert_frame_equal(got.astype(int), expected)
print("PASS: 2018 quarters match notebook 08; keys and target contract pass.")

,loans,positives
ORIGINATION QUARTER,,
Q1,11884,63
Q2,11815,264
Q3,11771,406
Q4,11947,489


AssertionError: DataFrame.index are different

Attribute "dtype" are different
[left]:  <StringDtype(na_value=<NA>)>
[right]: <StringDtype(na_value=nan)>

## 3. Does eligibility selection vary across 2018 quarters?

Compare all cleaned originations with eligible modeled loans. A difference in quarter-specific retention could affect observed event rates. This is a denominator check; it does not rebuild eligibility or assign labels to excluded loans.

In [ ]:
origin_count = orig.groupby("ORIGINATION QUARTER").size().rename("clean_origination_loans")
eligible_count = q18.groupby("ORIGINATION QUARTER").size().rename("eligible_24m_loans")
retention = pd.concat([origin_count, eligible_count], axis=1)
retention["not_in_modeling_artifact"] = retention["clean_origination_loans"] - retention["eligible_24m_loans"]
retention["retention_pct"] = 100 * retention["eligible_24m_loans"] / retention["clean_origination_loans"]
display(retention.round(3))
assert retention["clean_origination_loans"].sum() == 50000
assert retention["eligible_24m_loans"].sum() == 47417
assert retention["not_in_modeling_artifact"].ge(0).all()
print("Note: this quantifies exclusions, but the exact reason for each excluded loan requires the pre-filter target audit if retained.")

## 4. Which validated event component changes?

The composite event includes 90+ delinquency and qualifying terminal credit events. Count 90+ events, terminal-only events, and overlap separately. This determines whether the quarterly pattern is driven by the serious-delinquency component or by the terminal component.

In [ ]:
q18["serious"] = q18["TARGET_90_PLUS"].eq(1)
q18["terminal"] = q18["FIRST_TERMINAL_CREDIT_EVENT_PERIOD"].notna()
q18["terminal_only"] = q18["terminal"] & ~q18["serious"]
q18["both"] = q18["terminal"] & q18["serious"]
components = q18.groupby("ORIGINATION QUARTER").agg(
    loans=("LOAN IDENTIFIER","size"), composite=("TARGET_COMPOSITE_CREDIT_EVENT","sum"),
    serious=("serious","sum"), terminal_only=("terminal_only","sum"), both=("both","sum"))
components["event_rate_pct"] = 100 * components["composite"] / components["loans"]
display(components.round(3))
assert components["composite"].eq(components["serious"] + components["terminal_only"]).all()
assert q18["serious"].eq(q18["FIRST_SERIOUS_DELINQUENCY_PERIOD"].notna()).all()
print("PASS: composite events reconcile to 90+ plus terminal-only events.")

## 5. Inspect observation and first-event timing

Compare follow-up audit fields by origination quarter, then tabulate the *calendar year* of the first documented 90+ event. Later-originated loans' 24-month windows overlap different calendar periods. This table describes timing; it cannot by itself attribute the pattern to any economic or reporting cause.

In [ ]:
follow = q18.groupby("ORIGINATION QUARTER").agg(
    median_observed_months=("OBSERVED_MONTHS","median"),
    min_observed_months=("OBSERVED_MONTHS","min"),
    max_observed_months=("OBSERVED_MONTHS","max"),
    first_observed_not_zero=("FIRST_OBSERVED_MONTH",lambda s: int(s.ne(0).sum())),
    last_observed_before_23=("LAST_OBSERVED_MONTH",lambda s: int(s.lt(23).sum())))
display(follow)

events = q18.loc[q18["serious"], ["ORIGINATION QUARTER", "FIRST_SERIOUS_DELINQUENCY_PERIOD"]].copy()
events["first_event_year"] = pd.to_numeric(
    events["FIRST_SERIOUS_DELINQUENCY_PERIOD"].astype("string").str[:4], errors="coerce")
event_calendar = pd.crosstab(events["ORIGINATION QUARTER"], events["first_event_year"])
display(event_calendar)
assert event_calendar.to_numpy().sum() == int(q18["serious"].sum())
print("PASS: first serious-event calendar counts reconcile.")

## 6. Interpretation to record after local execution

Record (a) whether retention varies materially by quarter, (b) which target component accounts for the change, and (c) whether follow-up or event timing suggests a further audit. If the pattern remains unexplained, inspect the saved pre-filter target tables and source performance codes next. **Do not infer a causal explanation from this notebook, alter the established target, or lock the half-year analysis solely because the counts reconcile.**